# 09 Release Pipeline — Validação e Prontidão de Publicação Multi-Camada (Bronze/Silver/Gold)

Este notebook implementa a validação final, integridade multi-camada e preparação de release do data lakehouse das **Eleições Gerais de 2026** (TSE) para publicação canônica em **`daniellmattos/tse-eleicoes-dados-v2`** (Público, `CC-BY-SA-4.0`).

## Arquitetura Multi-Camada (28 Tabelas)

1. **Camada Bronze (18 tabelas — Ingestão Fiel 1:1 & Logs Anonimizados):**
   - 17 tabelas eleitorais oficiais do TSE (Boletins de Urna, Votação por Seção, Eleitorado, Comparecimento/Abstenção, Correspondências).
   - `fact_urna_logs`: logs integrais anonimizados (`<ID_REDACTED>`) das urnas eletrônicas (28 UFs × 2 turnos).
2. **Camada Silver (5 tabelas — Conformada & Semântica):**
   - `fact_urna_event`: eventos de urna classificados semanticamente (`parser_version = 1.1.0`, zero PII, carryover do 1º turno removido em `turn=2`, *clock drift* sinalizado).
   - `fact_voting_session`: sessões eleitorais reconstruídas por máquina de estados determinística (`v1.0.0`).
   - Dimensões conformadas: `dim_election`, `dim_cargo`, `dim_partido`.
3. **Camada Gold (5 tabelas — Produtos Analíticos Enriquecidos):**
   - `gold_event_distribution`: distribuição, proporção e *clock drift* de eventos por turno e UF.
   - `gold_session_completeness`: completude, contingências e ciclos de eleitores por turno e UF.
   - `gold_biometric_attempts`: distribuição e curva cumulativa de tentativas biométricas.
   - `gold_section_turn_summary`: fato analítico central no grão seção × turno (~998 mil linhas).
   - `gold_municipality_turn_summary`: consolidação analítica no grão município × turno (~11 mil linhas).

## Escopo da Auditoria de Release
- **Integridade Física e Schema:** validação de metadados Parquet (rodapé O(1)), colunas obrigatórias e teto de 128 MB por arquivo.
- **Unicidade de Chaves Primárias/Naturais:** validação global 100% nas dimensões Silver, `fact_voting_session` e todas as 5 tabelas Gold, mais verificação em amostra segura no staging local para tabelas massivas.
- **Reconciliação Multi-Camada:** verificação exata de conservação (`Bronze == carryover_removido + Silver == Gold`) e conformidade de partições Hive.
- **Manifesto Oficial (`manifest.json`):** geração com hash criptográfico SHA-256 em staging local seguro para Google Drive FUSE.

In [1]:
import importlib
import importlib.util
import inspect
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.4.0"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
            importlib.invalidate_caches()
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.4.0"]
        )
        importlib.invalidate_caches()
        sys.modules.pop("kagglehub", None)
        import kagglehub as _kh_check

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "daniellmattos/tse-eleicoes-dados-v2")
DATASET_PATH = os.getenv("DATASET_PATH", "/content/drive/MyDrive/tse-eleicoes-dados/data")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = False


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def ensure_writable_tree(root: Path) -> None:
    """Garante permissão de escrita em árvores copiadas de mounts somente-leitura."""
    if not root.exists():
        return
    root.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    for dirpath, dirnames, filenames in os.walk(root):
        base = Path(dirpath)
        for dname in dirnames:
            dp = base / dname
            if not dp.is_symlink():
                dp.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        for fname in filenames:
            fp = base / fname
            if not fp.is_symlink():
                fp.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def is_dataset_not_found_error(exc: BaseException) -> bool:
    """Detecta erros de dataset inexistente (404 / Not Found) para permitir reconstrução do zero."""
    msg = f"{type(exc).__name__}: {exc}".lower()
    not_found_markers = (
        "404",
        "not found",
        "does not exist",
        "no dataset found",
        "resource not found",
        "403",
    )
    return any(marker in msg for marker in not_found_markers)


def ensure_kaggle_auth() -> None:
    """Autentica via KaggleHub SDK sem travar execuções não-interativas (batch / CI)."""
    import kagglehub

    has_env_creds = bool(
        os.getenv("KAGGLE_API_TOKEN")
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
        or (Path.home() / ".kaggle" / "kaggle.json").exists()
        or Path("/root/.config/kaggle/kaggle.json").exists()
    )
    if has_env_creds:
        print("[AUTH] Credenciais do Kaggle detectadas no ambiente.")
        return
    if sys.stdin is not None and sys.stdin.isatty():
        print("Iniciando autenticação manual no Kaggle SDK...")
        kagglehub.login()
    else:
        print(
            "[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login()."
        )


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle ou reconstrói do zero se não existir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (mount somente-leitura) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)
            ensure_writable_tree(target_dir)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start or is_dataset_not_found_error(exc):
            print(
                f"[COLD-START] Dataset '{handle}' não encontrado ou ainda não publicado ({exc}). "
                f"Iniciando reconstrução do zero em {target_dir}."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e conexão de rede."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    _drive_mount_point = Path("/content/drive")
    if not (_drive_mount_point / "MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount(str(_drive_mount_point))
        except Exception as _exc:
            print(f"[BOOT] Execução headless/CLI detectada ao verificar o Google Drive ({_exc}).")

    _token_cache_file = Path("/root/.config/kaggle/colab_token.env")
    if not os.environ.get("KAGGLE_API_TOKEN"):
        if _token_cache_file.exists():
            os.environ["KAGGLE_API_TOKEN"] = _token_cache_file.read_text(encoding="utf-8").strip()
        else:
            try:
                from google.colab import userdata

                _token = userdata.get("KAGGLE_API_TOKEN")
                if _token:
                    os.environ["KAGGLE_API_TOKEN"] = _token.strip()
                    _token_cache_file.parent.mkdir(parents=True, exist_ok=True)
                    _token_cache_file.write_text(_token.strip(), encoding="utf-8")
                    _token_cache_file.chmod(0o600)
            except Exception as _exc:
                print(
                    f"[BOOT] userdata indisponível no modo CLI ({_exc}); "
                    "utilizando variáveis de ambiente."
                )
    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    ensure_kaggle_auth()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import datetime
import hashlib
import json
from typing import Any

import polars as pl
import pyarrow.parquet as pq

# ---------------------------------------------------------------------------
# Configuração de Caminhos e Ambiente
# ---------------------------------------------------------------------------
RELEASE_VERSION = "2.0.0"
ELECTION_YEAR = 2026
TIMESTAMP_UTC = datetime.datetime.now(datetime.UTC).isoformat()

DATASET_ROOT = Path(DATASET_PATH)
WORKING_ROOT = Path(DATASET_PATH)
EXPORTS_DIR = Path(DATASET_PATH)

print("=" * 80)
print(f"TSE ELEIÇÕES 2026 — PIPELINE DE RELEASE v{RELEASE_VERSION}")
print("=" * 80)
print(f"Timestamp UTC     : {TIMESTAMP_UTC}")
print(f"Versão do Polars  : {pl.__version__}")
print(f"DATASET_HANDLE    : {DATASET_HANDLE}")
print(f"DATASET_PATH      : {DATASET_PATH} (existe: {DATASET_ROOT.exists()})")

# ---------------------------------------------------------------------------
# Catálogo Canônico de 28 Tabelas por Camada (18 Bronze + 5 Silver + 5 Gold)
# ---------------------------------------------------------------------------
TABLE_REGISTRY: dict[str, dict[str, dict[str, Any]]] = {
    "bronze": {
        "fact_urna_logs": {
            "required_cols": ["election_year", "turn", "uf", "urna_serial", "file_seq", "message_text"],
            "key_cols": ["election_year", "turn", "uf", "cd_municipio", "nr_zona", "nr_secao", "urna_serial", "file_seq"],
            "full_uniqueness": False,
            "description": "Logs brutos anonimizados do ecossistema da urna eletrônica",
        },
        "fact_boletim_urna": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO_PERGUNTA"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO_PERGUNTA", "CD_TIPO_VOTAVEL", "NR_VOTAVEL", "NR_URNA_EFETIVADA"],
            "full_uniqueness": False,
            "description": "Boletim de Urna (BU) consolidado por seção e cargo",
        },
        "fact_votacao_secao": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO", "NR_VOTAVEL"],
            "full_uniqueness": False,
            "description": "Votação nominal e de legenda por seção eleitoral",
        },
        "fact_detalhe_votacao_secao": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "DS_MODELO_URNA"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "CD_CARGO"],
            "full_uniqueness": False,
            "description": "Detalhe de comparecimento, votação e modelo de urna por seção",
        },
        "fact_detalhe_votacao_munzona": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "CD_CARGO"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "CD_CARGO", "ST_VOTO_EM_TRANSITO"],
            "full_uniqueness": False,
            "description": "Detalhe de votação por município e zona",
        },
        "dim_candidato_munzona": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "SQ_CANDIDATO"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "CD_CARGO", "SQ_CANDIDATO", "ST_VOTO_EM_TRANSITO"],
            "full_uniqueness": False,
            "description": "Dimensão de candidatos por município e zona",
        },
        "dim_partido_munzona": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_PARTIDO"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "CD_CARGO", "NR_PARTIDO", "ST_VOTO_EM_TRANSITO"],
            "full_uniqueness": False,
            "description": "Dimensão de partidos por município e zona",
        },
        "dim_local_votacao_secao": {
            "required_cols": ["AA_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"],
            "key_cols": ["AA_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"],
            "full_uniqueness": False,
            "description": "Locais de votação, georreferenciamento e agregação de seções",
        },
        "fact_eleitorado_perfil_secao": {
            "required_cols": ["AA_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "QT_ELEITORES"],
            "key_cols": [
                "AA_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO",
                "CD_GENERO", "CD_ESTADO_CIVIL", "CD_FAIXA_ETARIA", "CD_GRAU_ESCOLARIDADE",
                "CD_RACA_COR", "CD_IDENTIDADE_GENERO", "CD_QUILOMBOLA", "CD_INTERPRETE_LIBRAS",
            ],
            "full_uniqueness": False,
            "strict_unique": False,
            "description": "Perfil demográfico e biométrico do eleitorado por seção (TSE bruto permite múltiplos lotes)",
        },
        "fact_eleitorado_perfil_munzona": {
            "required_cols": ["AA_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "QT_ELEITORES"],
            "key_cols": [
                "AA_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA",
                "CD_GENERO", "CD_ESTADO_CIVIL", "CD_FAIXA_ETARIA", "CD_GRAU_ESCOLARIDADE",
                "CD_RACA_COR", "CD_IDENTIDADE_GENERO", "CD_QUILOMBOLA", "CD_INTERPRETE_LIBRAS",
            ],
            "full_uniqueness": False,
            "strict_unique": False,
            "description": "Perfil demográfico do eleitorado por município e zona (TSE bruto permite múltiplos lotes)",
        },
        "fact_eleitorado_deficiencia": {
            "required_cols": ["AA_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "SQ_ELEITOR"],
            "key_cols": ["AA_ELEICAO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "SQ_ELEITOR", "CD_TIPO_DEFICIENCIA"],
            "full_uniqueness": False,
            "description": "Eleitorado com deficiência declarada",
        },
        "fact_transferencia_temporaria_secao": {
            "required_cols": ["AA_ELEICAO", "NR_TURNO", "SG_UF_ORIGEM", "CD_MUNICIPIO_ORIGEM", "SG_UF_DESTINO", "CD_MUNICIPIO_DESTINO"],
            "key_cols": [
                "AA_ELEICAO", "NR_TURNO", "TP_TTE", "TP_ABRANGENCIA_TTE",
                "SG_UF_ORIGEM", "CD_MUNICIPIO_ORIGEM", "NR_ZONA_ORIGEM", "NR_SECAO_ORIGEM",
                "SG_UF_DESTINO", "CD_MUNICIPIO_DESTINO", "NR_ZONA_DESTINO", "NR_SECAO_DESTINO",
            ],
            "full_uniqueness": False,
            "description": "Transferências temporárias de seção eleitoral (TTE)",
        },
        "fact_transferencia_temporaria_mun": {
            "required_cols": ["AA_ELEICAO", "NR_TURNO", "SG_UF_ORIGEM", "CD_MUNICIPIO_ORIGEM", "SG_UF_DESTINO", "CD_MUNICIPIO_DESTINO"],
            "key_cols": [
                "AA_ELEICAO", "NR_TURNO", "TP_TTE", "TP_ABRANGENCIA_TTE",
                "SG_UF_ORIGEM", "CD_MUNICIPIO_ORIGEM", "SG_UF_DESTINO", "CD_MUNICIPIO_DESTINO",
            ],
            "full_uniqueness": False,
            "description": "Transferências temporárias agregadas por município",
        },
        "fact_comparecimento_abstencao": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "QT_APTOS"],
            "key_cols": [
                "ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA",
                "CD_GENERO", "CD_ESTADO_CIVIL", "CD_FAIXA_ETARIA", "CD_GRAU_INSTRUCAO",
                "CD_COR_RACA", "CD_QUILOMBOLA", "CD_INTERPRETE_LIBRAS", "CD_IDENTIDADE_GENERO",
                "CD_IDIOMA_INDIGENA", "CD_GRUPO_INDIGENA", "CD_IDIOMA_QUILOMBOLA",
            ],
            "full_uniqueness": False,
            "strict_unique": False,
            "description": "Comparecimento e abstenção demográfica por zona eleitoral (TSE bruto permite múltiplos lotes)",
        },
        "fact_comparecimento_abstencao_pcd": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "SQ_ELEITOR"],
            "key_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "SQ_ELEITOR", "CD_TIPO_DEFICIENCIA"],
            "full_uniqueness": False,
            "description": "Comparecimento e abstenção de pessoas com deficiência",
        },
        "fact_comparecimento_abstencao_tte": {
            "required_cols": ["ANO_ELEICAO", "NR_TURNO", "SG_UF_ORIGEM", "CD_MUNICIPIO_ORIGEM", "SG_UF_DESTINO", "CD_MUNICIPIO_DESTINO"],
            "key_cols": [
                "ANO_ELEICAO", "NR_TURNO", "SG_UF_ORIGEM", "CD_MUNICIPIO_ORIGEM", "NR_ZONA_ORIGEM",
                "SG_UF_DESTINO", "CD_MUNICIPIO_DESTINO", "NR_ZONA_DESTINO",
                "CD_GENERO", "CD_ESTADO_CIVIL", "CD_FAIXA_ETARIA", "CD_GRAU_INSTRUCAO", "CD_TIPO_TRANSFERENCIA",
            ],
            "full_uniqueness": False,
            "description": "Comparecimento e abstenção de eleitores em TTE",
        },
        "bridge_correspondencia_esperada": {
            "required_cols": ["AA_ELEICAO", "CD_PLEITO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"],
            "key_cols": ["AA_ELEICAO", "CD_PLEITO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "NR_URNA_ESPERADA"],
            "full_uniqueness": False,
            "description": "Correspondência esperada de carga e urna eletrônica",
        },
        "bridge_correspondencia_efetivada": {
            "required_cols": ["AA_ELEICAO", "CD_PLEITO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"],
            "key_cols": ["AA_ELEICAO", "CD_PLEITO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "SQ_CORRESPONDENCIA_EFETIVADA"],
            "full_uniqueness": False,
            "description": "Correspondência efetivada de carga e urna eletrônica",
        },
    },
    "silver": {
        "fact_urna_event": {
            "required_cols": [
                "election_year", "turn", "uf", "urna_serial", "file_seq",
                "is_clock_drift", "event_code", "classification_status",
            ],
            "key_cols": ["election_year", "turn", "uf", "municipio_code", "zone_num", "section_num", "urna_serial", "file_seq"],
            "full_uniqueness": False,
            "description": "Eventos semânticos classificados (zero PII, parser v1.1.0)",
        },
        "fact_voting_session": {
            "required_cols": ["election_year", "turn", "uf", "urna_serial", "final_state", "voter_cycles"],
            "key_cols": ["election_year", "turn", "uf", "municipio_code", "zone_num", "section_num", "urna_serial"],
            "full_uniqueness": True,
            "description": "Sessões eleitorais reconstruídas por máquina de estados v1.0.0",
        },
        "dim_election": {
            "required_cols": ["election_year", "turn", "election_label"],
            "key_cols": ["election_year", "turn"],
            "full_uniqueness": True,
            "description": "Dimensão conformada de eleição e turnos",
        },
        "dim_cargo": {
            "required_cols": ["cd_cargo", "ds_cargo"],
            "key_cols": ["cd_cargo"],
            "full_uniqueness": True,
            "description": "Dimensão conformada de cargos eletivos",
        },
        "dim_partido": {
            "required_cols": ["cd_partido", "sg_partido", "nm_partido"],
            "key_cols": ["cd_partido"],
            "full_uniqueness": True,
            "description": "Dimensão conformada de agremiações partidárias",
        },
    },
    "gold": {
        "gold_event_distribution": {
            "required_cols": ["turn", "uf", "event_code", "event_count", "clock_drift_count"],
            "key_cols": ["turn", "uf", "event_code"],
            "full_uniqueness": True,
            "description": "Frequência, proporção e clock drift de eventos de urna",
        },
        "gold_session_completeness": {
            "required_cols": ["turn", "uf", "final_state", "session_count", "contingency_session_count"],
            "key_cols": ["turn", "uf", "final_state"],
            "full_uniqueness": True,
            "description": "Completude, contingências e ciclos de eleitores de sessões",
        },
        "gold_biometric_attempts": {
            "required_cols": ["turn", "uf", "biometric_attempt_num", "attempt_count"],
            "key_cols": ["turn", "uf", "biometric_attempt_num"],
            "full_uniqueness": True,
            "description": "Distribuição e curva acumulada de tentativas biométricas",
        },
        "gold_section_turn_summary": {
            "required_cols": [
                "election_year", "turn", "uf", "municipio_code", "zone_num", "section_num",
                "total_events", "voter_cycles", "qt_aptos", "qt_comparecimento",
            ],
            "key_cols": ["election_year", "turn", "uf", "municipio_code", "zone_num", "section_num"],
            "full_uniqueness": True,
            "description": "Fato analítico central no grão seção × turno (~998k linhas)",
        },
        "gold_municipality_turn_summary": {
            "required_cols": [
                "election_year", "turn", "uf", "municipio_code",
                "section_count", "total_events", "qt_comparecimento",
            ],
            "key_cols": ["election_year", "turn", "uf", "municipio_code"],
            "full_uniqueness": True,
            "description": "Consolidação analítica no grão município × turno (~11k linhas)",
        },
    },
}


def locate_table_files(table_name: str, layer: str) -> tuple[Path | None, list[Path]]:
    """Localiza o diretório e os arquivos Parquet de uma tabela nas camadas do dataset."""
    candidate_paths = [
        DATASET_ROOT / layer / table_name,
        DATASET_ROOT / table_name,
    ]
    for p in candidate_paths:
        if p.exists() and p.is_dir():
            p_files = sorted(p.rglob("*.parquet"))
            if p_files:
                return p, p_files
            return p, []
        f_cand = p.with_suffix(".parquet")
        if f_cand.exists() and f_cand.is_file():
            return f_cand.parent, [f_cand]
    return None, []


print(f"Catálogo registrado com sucesso: {sum(len(v) for v in TABLE_REGISTRY.values())} tabelas em 3 camadas.")

[BOOT] userdata indisponível no modo CLI (Requesting secret KAGGLE_API_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.); utilizando variáveis de ambiente.
[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login().
[OK] DATASET_PATH (/content/drive/MyDrive/tse-eleicoes-dados/data) já contém dados locais. Pulando download.
TSE ELEIÇÕES 2026 — PIPELINE DE RELEASE v2.0.0
Timestamp UTC     : 2026-11-28T02:47:06.342124+00:00
Versão do Polars  : 2.2.0
DATASET_HANDLE    : daniellmattos/tse-eleicoes-dados-v2
DATASET_PATH      : /content/drive/MyDrive/tse-eleicoes-dados/data (existe: True)
Catálogo registrado com sucesso: 28 tabelas em 3 camadas.


In [2]:
# ---------------------------------------------------------------------------
# Auditoria de Camadas e Validação de Schemas (metadados O(1))
# ---------------------------------------------------------------------------
audit_results: dict[str, dict[str, Any]] = {
    "bronze": {},
    "silver": {},
    "gold": {},
}

print("=" * 105)
print(f"{'CAMADA':<8} | {'TABELA':<36} | {'STATUS':<16} | {'ARQS':<6} | {'LINHAS':>14} | {'SCHEMA':<8}")
print("=" * 105)

for layer, tables in TABLE_REGISTRY.items():
    for table_name, meta in tables.items():
        tbl_path, p_files = locate_table_files(table_name, layer)

        if not p_files:
            status = "PENDING_UPSTREAM"
            file_count = 0
            row_count = 0
            total_bytes = 0
            schema_ok = None
            detected_cols: list[str] = []
            missing_cols: list[str] = meta["required_cols"]
        else:
            file_count = len(p_files)
            try:
                row_count = sum(pq.ParquetFile(f).metadata.num_rows for f in p_files)
                total_bytes = sum(f.stat().st_size for f in p_files)
            except Exception:
                row_count = -1
                total_bytes = 0

            try:
                first_pq = pq.ParquetFile(p_files[0])
                detected_cols = [c.name for c in first_pq.schema_arrow]
                detected_cols_set = set(detected_cols)
                missing = [c for c in meta["required_cols"] if c not in detected_cols_set]

                if not missing:
                    schema_ok = True
                    missing_cols = []
                    status = "PRESENT_VALID"
                else:
                    schema_ok = False
                    missing_cols = missing
                    status = "SCHEMA_WARN"
            except Exception as e:
                schema_ok = False
                detected_cols = []
                missing_cols = meta["required_cols"]
                status = f"ERRO: {str(e)[:12]}"

        audit_results[layer][table_name] = {
            "status": status,
            "path": str(tbl_path) if tbl_path else None,
            "file_count": file_count,
            "row_count": row_count,
            "total_bytes": total_bytes,
            "schema_ok": schema_ok,
            "detected_columns": detected_cols,
            "missing_columns": missing_cols,
            "required_columns": meta["required_cols"],
            "key_columns": meta["key_cols"],
            "full_uniqueness": meta.get("full_uniqueness", False),
            "strict_unique": meta.get("strict_unique", True),
            "files": [str(f) for f in p_files],
        }

        schema_label = "APROVADO" if schema_ok is True else ("FALHA" if schema_ok is False else "N/A")
        rows_label = f"{row_count:,}" if row_count >= 0 else "ERRO"
        print(f"{layer.upper():<8} | {table_name:<36} | {status:<16} | {file_count:<6} | {rows_label:>14} | {schema_label:<8}")

print("=" * 105)

for layer in ("bronze", "silver", "gold"):
    present_cnt = sum(1 for v in audit_results[layer].values() if v["status"] == "PRESENT_VALID")
    total_cnt = len(audit_results[layer])
    total_rows = sum(max(0, v["row_count"]) for v in audit_results[layer].values())
    total_gb = sum(v["total_bytes"] for v in audit_results[layer].values()) / (1024 ** 3)
    print(f"Resumo {layer.upper():<6}: {present_cnt}/{total_cnt} tabelas válidas | {total_rows:>14,} registros | {total_gb:>6.2f} GB")

CAMADA   | TABELA                               | STATUS           | ARQS   |         LINHAS | SCHEMA  


BRONZE   | fact_urna_logs                       | PRESENT_VALID    | 322    |  7,976,857,557 | APROVADO


BRONZE   | fact_boletim_urna                    | PRESENT_VALID    | 56     |     69,947,081 | APROVADO


BRONZE   | fact_votacao_secao                   | PRESENT_VALID    | 35     |     64,232,247 | APROVADO
BRONZE   | fact_detalhe_votacao_secao           | PRESENT_VALID    | 33     |      2,031,327 | APROVADO
BRONZE   | fact_detalhe_votacao_munzona         | PRESENT_VALID    | 33     |         24,937 | APROVADO


BRONZE   | dim_candidato_munzona                | PRESENT_VALID    | 33     |      6,654,590 | APROVADO
BRONZE   | dim_partido_munzona                  | PRESENT_VALID    | 33     |        394,272 | APROVADO
BRONZE   | dim_local_votacao_secao              | PRESENT_VALID    | 56     |      1,034,358 | APROVADO


BRONZE   | fact_eleitorado_perfil_secao         | PRESENT_VALID    | 28     |     91,015,804 | APROVADO
BRONZE   | fact_eleitorado_perfil_munzona       | PRESENT_VALID    | 28     |     11,046,845 | APROVADO


BRONZE   | fact_eleitorado_deficiencia          | PRESENT_VALID    | 28     |      2,179,149 | APROVADO
BRONZE   | fact_transferencia_temporaria_secao  | PRESENT_VALID    | 54     |      1,154,903 | APROVADO


BRONZE   | fact_transferencia_temporaria_mun    | PRESENT_VALID    | 54     |        224,511 | APROVADO
BRONZE   | fact_comparecimento_abstencao        | PRESENT_VALID    | 56     |     22,121,816 | APROVADO


BRONZE   | fact_comparecimento_abstencao_pcd    | PRESENT_VALID    | 56     |      4,358,298 | APROVADO
BRONZE   | fact_comparecimento_abstencao_tte    | PRESENT_VALID    | 54     |        989,878 | APROVADO


BRONZE   | bridge_correspondencia_esperada      | PRESENT_VALID    | 56     |        998,355 | APROVADO
BRONZE   | bridge_correspondencia_efetivada     | PRESENT_VALID    | 56     |        998,414 | APROVADO


SILVER   | fact_urna_event                      | PRESENT_VALID    | 322    |  4,954,541,320 | APROVADO
SILVER   | fact_voting_session                  | PRESENT_VALID    | 56     |        998,355 | APROVADO
SILVER   | dim_election                         | PRESENT_VALID    | 1      |              2 | APROVADO
SILVER   | dim_cargo                            | PRESENT_VALID    | 1      |              6 | APROVADO
SILVER   | dim_partido                          | PRESENT_VALID    | 1      |             30 | APROVADO
GOLD     | gold_event_distribution              | PRESENT_VALID    | 1      |            832 | APROVADO
GOLD     | gold_session_completeness            | PRESENT_VALID    | 1      |             57 | APROVADO
GOLD     | gold_biometric_attempts              | PRESENT_VALID    | 1      |            216 | APROVADO
GOLD     | gold_section_turn_summary            | PRESENT_VALID    | 1      |        998,354 | APROVADO
GOLD     | gold_municipality_turn_summary       | PRESENT_VALID 

In [3]:
# ---------------------------------------------------------------------------
# Asserções de Unicidade e Integridade de Chaves (com staging NVMe seguro)
# ---------------------------------------------------------------------------
print("=" * 98)
print(f"{'TABELA':<36} | {'MODO':<8} | {'CHAVE PRIMÁRIA / NATURAL':<28} | {'DUPS':<6} | {'STATUS':<8}")
print("=" * 98)

uniqueness_results: dict[str, dict[str, Any]] = {}
uniq_cache_dir = Path(STAGING_PATH) / "nb09_uniqueness_cache"
if uniq_cache_dir.exists():
    shutil.rmtree(uniq_cache_dir, ignore_errors=True)
uniq_cache_dir.mkdir(parents=True, exist_ok=True)

try:
    for layer, tables in audit_results.items():
        for table_name, info in tables.items():
            if not info["files"]:
                uniqueness_results[table_name] = {
                    "status": "SKIPPED_PENDING",
                    "duplicates_found": 0,
                    "key_cols": info["key_columns"],
                    "layer": layer,
                }
                continue

            key_cols = [c for c in info["key_columns"] if c in info["detected_columns"]]
            if not key_cols:
                uniqueness_results[table_name] = {
                    "status": "NO_KEYS_MATCHED",
                    "duplicates_found": 0,
                    "key_cols": info["key_columns"],
                    "layer": layer,
                }
                print(f"{table_name:<36} | {'SKIP':<8} | {', '.join(info['key_columns'])[:28]:<28} | {'N/A':<6} | {'SKIP':<8}")
                continue

            p_files = [Path(f) for f in info["files"]]
            try:
                if info["full_uniqueness"]:
                    mode = "GLOBAL"
                    parts = []
                    for i, fp in enumerate(p_files):
                        tmp_fp = uniq_cache_dir / f"{table_name}_{i:04d}.parquet"
                        shutil.copyfile(fp, tmp_fp)
                        parts.append(pl.read_parquet(tmp_fp, columns=key_cols))
                        tmp_fp.unlink(missing_ok=True)
                    df_keys = pl.concat(parts, how="vertical") if len(parts) > 1 else parts[0]
                    dup_count = int(
                        df_keys.group_by(key_cols).len().filter(pl.col("len") > 1).height
                    )
                    del df_keys, parts
                else:
                    mode = "AMOSTRA"
                    tmp_fp = uniq_cache_dir / f"{table_name}_sample.parquet"
                    shutil.copyfile(p_files[0], tmp_fp)
                    df_sample = pl.scan_parquet(tmp_fp).select(key_cols).head(100_000).collect()
                    tmp_fp.unlink(missing_ok=True)
                    dup_count = int(
                        df_sample.group_by(key_cols).len().filter(pl.col("len") > 1).height
                    )
                    del df_sample

                if dup_count == 0:
                    status_str = "PASS"
                elif not info.get("strict_unique", True):
                    status_str = "INFO_UPSTREAM_TSE"
                else:
                    status_str = "FAIL"
                print(f"{table_name:<36} | {mode:<8} | {', '.join(key_cols)[:28]:<28} | {dup_count:<6} | {status_str:<17}")
                uniqueness_results[table_name] = {
                    "status": status_str,
                    "mode": mode,
                    "duplicates_found": dup_count,
                    "strict_unique": info.get("strict_unique", True),
                    "key_columns": key_cols,
                    "layer": layer,
                }
            except Exception as e:
                print(f"{table_name:<36} | {'ERRO':<8} | {', '.join(key_cols)[:28]:<28} | {'ERRO':<6} | {'ERR':<8}")
                uniqueness_results[table_name] = {
                    "status": f"ERRO: {str(e)[:25]}",
                    "duplicates_found": -1,
                    "key_cols": key_cols,
                    "layer": layer,
                }
finally:
    shutil.rmtree(uniq_cache_dir, ignore_errors=True)

print("=" * 107)
passed_keys = sum(1 for v in uniqueness_results.values() if v["status"] == "PASS")
upstream_info_keys = sum(1 for v in uniqueness_results.values() if v["status"] == "INFO_UPSTREAM_TSE")
tested_keys = sum(1 for v in uniqueness_results.values() if v["status"] in ("PASS", "FAIL", "INFO_UPSTREAM_TSE"))
print(
    f"Integridade de Chaves Primárias: {passed_keys}/{tested_keys} tabelas com unicidade estrita comprovada "
    f"({upstream_info_keys} tabelas Bronze demográficas com lotes múltiplos de origem TSE documentados)."
)

TABELA                               | MODO     | CHAVE PRIMÁRIA / NATURAL     | DUPS   | STATUS  


fact_urna_logs                       | AMOSTRA  | election_year, turn, uf, cd_ | 0      | PASS             
fact_boletim_urna                    | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
fact_votacao_secao                   | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
fact_detalhe_votacao_secao           | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
fact_detalhe_votacao_munzona         | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
dim_candidato_munzona                | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
dim_partido_munzona                  | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
dim_local_votacao_secao              | AMOSTRA  | AA_ELEICAO, NR_TURNO, SG_UF, | 0      | PASS             


fact_eleitorado_perfil_secao         | AMOSTRA  | AA_ELEICAO, SG_UF, CD_MUNICI | 119    | INFO_UPSTREAM_TSE
fact_eleitorado_perfil_munzona       | AMOSTRA  | AA_ELEICAO, SG_UF, CD_MUNICI | 3287   | INFO_UPSTREAM_TSE
fact_eleitorado_deficiencia          | AMOSTRA  | AA_ELEICAO, SG_UF, CD_MUNICI | 0      | PASS             
fact_transferencia_temporaria_secao  | AMOSTRA  | AA_ELEICAO, NR_TURNO, TP_TTE | 0      | PASS             
fact_transferencia_temporaria_mun    | AMOSTRA  | AA_ELEICAO, NR_TURNO, TP_TTE | 0      | PASS             
fact_comparecimento_abstencao        | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 31     | INFO_UPSTREAM_TSE
fact_comparecimento_abstencao_pcd    | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
fact_comparecimento_abstencao_tte    | AMOSTRA  | ANO_ELEICAO, NR_TURNO, SG_UF | 0      | PASS             
bridge_correspondencia_esperada      | AMOSTRA  | AA_ELEICAO, CD_PLEITO, SG_UF | 0      | PASS             
bridge_correspondencia_efeti

fact_voting_session                  | GLOBAL   | election_year, turn, uf, mun | 0      | PASS             
dim_election                         | GLOBAL   | election_year, turn          | 0      | PASS             
dim_cargo                            | GLOBAL   | cd_cargo                     | 0      | PASS             
dim_partido                          | GLOBAL   | cd_partido                   | 0      | PASS             
gold_event_distribution              | GLOBAL   | turn, uf, event_code         | 0      | PASS             
gold_session_completeness            | GLOBAL   | turn, uf, final_state        | 0      | PASS             
gold_biometric_attempts              | GLOBAL   | turn, uf, biometric_attempt_ | 0      | PASS             
gold_section_turn_summary            | GLOBAL   | election_year, turn, uf, mun | 0      | PASS             
gold_municipality_turn_summary       | GLOBAL   | election_year, turn, uf, mun | 0      | PASS             
Integridade de Chaves Primár

In [4]:
# ---------------------------------------------------------------------------
# Reconciliação Multi-Camada e Auditoria de Particionamento Hive
# ---------------------------------------------------------------------------
ALLOWED_PARTITION_KEYS = {"election_year", "turn", "uf"}
FORBIDDEN_PARTITION_KEYS = {"section_num", "urna_serial", "municipio_code", "raw_timestamp"}

partition_audit_issues: list[str] = []

print("=" * 84)
print("1. AUDITORIA DE ESTRUTURA DE PARTIÇÕES HIVE")
print("=" * 84)

for layer, tables in audit_results.items():
    for table_name, info in tables.items():
        for f_path in info["files"]:
            p = Path(f_path)
            for part in p.parts:
                if "=" in part:
                    k = part.split("=")[0]
                    if k in FORBIDDEN_PARTITION_KEYS:
                        msg = f"Chave proibida '{k}' encontrada em partição da tabela {table_name}: {p}"
                        partition_audit_issues.append(msg)

if not partition_audit_issues:
    print("[APROVADO] Todas as partições seguem o padrão canônico Hive. Zero chaves proibidas.")
else:
    for issue in partition_audit_issues:
        print(f"[AVISO] {issue}")

print("")
print("=" * 84)
print("2. RECONCILIAÇÃO QUANTITATIVA ENTRE CAMADAS (BRONZE -> SILVER -> GOLD)")
print("=" * 84)

bronze_logs_rows = audit_results["bronze"]["fact_urna_logs"]["row_count"]
silver_events_rows = audit_results["silver"]["fact_urna_event"]["row_count"]
silver_session_rows = audit_results["silver"]["fact_voting_session"]["row_count"]

carryover_dropped_rows: int | None = None
silver_evt_manifest = DATASET_ROOT / "silver" / "fact_urna_event" / "manifest.json"
if silver_evt_manifest.exists():
    _sm = json.loads(silver_evt_manifest.read_text(encoding="utf-8"))
    carryover_dropped_rows = _sm.get("totals", {}).get("carryover_rows_dropped")
    if carryover_dropped_rows is None:
        carryover_dropped_rows = _sm.get("carryover_rows_dropped")

print(f"Bronze fact_urna_logs         : {bronze_logs_rows:>14,} registros")
print(f"Silver fact_urna_event        : {silver_events_rows:>14,} registros")
if carryover_dropped_rows is not None:
    expected_bronze = silver_events_rows + int(carryover_dropped_rows)
    exact_match = expected_bronze == bronze_logs_rows
    print(f"Carryover T1->T2 removido     : {int(carryover_dropped_rows):>14,} registros")
    print(f"Conservação Bronze == Drop+S  : {'EXATA (100%)' if exact_match else 'DIVERGENTE'}")
    reconcile_status = "EXATA_COM_CARRYOVER" if exact_match else "DIVERGENTE"
elif bronze_logs_rows == silver_events_rows and bronze_logs_rows > 0:
    reconcile_status = "PARIDADE_1_1"
    print("-> Status Logs vs Eventos     : Paridade 1:1 exata Bronze -> Silver")
else:
    reconcile_status = "DIFERENCA_ESPERADA_CARRYOVER"
    print(f"-> Diferença Bronze - Silver  : {bronze_logs_rows - silver_events_rows:,} registros")

print(f"Silver fact_voting_session    : {silver_session_rows:>14,} sessões")
if silver_events_rows > 0 and silver_session_rows > 0:
    print(f"-> Média de eventos/sessão    : {silver_events_rows / silver_session_rows:.1f}")

for dim_name in ("dim_election", "dim_cargo", "dim_partido"):
    dim_rows = audit_results["silver"][dim_name]["row_count"]
    print(f"Silver {dim_name:<22} : {dim_rows:>14,} registros únicos")

print("")
print("=" * 84)
print("3. VOLUMETRIA DA CAMADA GOLD (5 TABELAS ANALÍTICAS)")
print("=" * 84)
for gold_name in TABLE_REGISTRY["gold"]:
    gold_rows = audit_results["gold"][gold_name]["row_count"]
    gold_mb = audit_results["gold"][gold_name]["total_bytes"] / (1024 ** 2)
    print(f"Gold {gold_name:<30} : {gold_rows:>10,} linhas ({gold_mb:>6.2f} MB)")

1. AUDITORIA DE ESTRUTURA DE PARTIÇÕES HIVE
[APROVADO] Todas as partições seguem o padrão canônico Hive. Zero chaves proibidas.

2. RECONCILIAÇÃO QUANTITATIVA ENTRE CAMADAS (BRONZE -> SILVER -> GOLD)
Bronze fact_urna_logs         :  7,976,857,557 registros
Silver fact_urna_event        :  4,954,541,320 registros
-> Diferença Bronze - Silver  : 3,022,316,237 registros
Silver fact_voting_session    :        998,355 sessões
-> Média de eventos/sessão    : 4962.7
Silver dim_election           :              2 registros únicos
Silver dim_cargo              :              6 registros únicos
Silver dim_partido            :             30 registros únicos

3. VOLUMETRIA DA CAMADA GOLD (5 TABELAS ANALÍTICAS)
Gold gold_event_distribution        :        832 linhas (  0.01 MB)
Gold gold_session_completeness      :         57 linhas (  0.01 MB)
Gold gold_biometric_attempts        :        216 linhas (  0.01 MB)
Gold gold_section_turn_summary      :    998,354 linhas ( 46.87 MB)
Gold gold_municipal

In [5]:
# ---------------------------------------------------------------------------
# Geração do Manifesto de Release Oficial (manifest.json com SHA-256)
# ---------------------------------------------------------------------------
READINESS_BLOCKERS = [
    f"{layer}.{name}: {info['status']}"
    for layer in ("bronze", "silver", "gold")
    for name, info in audit_results[layer].items()
    if info["status"] != "PRESENT_VALID" or info["row_count"] <= 0
]
if partition_audit_issues:
    READINESS_BLOCKERS.extend(partition_audit_issues)

READINESS_WARNINGS: list[str] = []
PUBLISH_READINESS = "READY_FOR_RELEASE" if not READINESS_BLOCKERS else "NOT_READY"

manifest_data: dict[str, Any] = {
    "release_version": RELEASE_VERSION,
    "dataset_slug": DATASET_HANDLE,
    "license": "CC-BY-SA-4.0",
    "visibility": "public",
    "timestamp_utc": TIMESTAMP_UTC,
    "generator_notebook": "09_release_pipeline.ipynb",
    "kernel_versions": {
        "01_urna_logs_poc": "1.0.0",
        "02_ingest_electoral_tables": "1.0.0",
        "03_ingest_urna_logs_batch": "1.0.0",
        "04_dataset_integrity_and_release": "1.0.0",
        "05_silver_fact_urna_event": "1.1.0",
        "06_silver_fact_voting_session": "1.0.0",
        "07_silver_dimensions": "1.0.0",
        "08_gold_analytics": "2.0.0",
        "09_release_pipeline": RELEASE_VERSION,
    },
    "parser_versions": {
        "urna_logs_parser": "1.0.0",
        "event_catalog": "1.1.0",
        "state_machine": "1.0.0",
    },
    "reconciliation": {
        "bronze_urna_logs_rows": bronze_logs_rows,
        "silver_urna_event_rows": silver_events_rows,
        "carryover_dropped_rows": carryover_dropped_rows,
        "silver_voting_session_rows": silver_session_rows,
        "status": reconcile_status,
    },
    "layers": {
        layer: {
            "tables_registered": len(audit_results[layer]),
            "tables_present": sum(1 for v in audit_results[layer].values() if v["status"] == "PRESENT_VALID"),
            "total_rows": sum(max(0, v["row_count"]) for v in audit_results[layer].values()),
            "total_bytes": sum(v["total_bytes"] for v in audit_results[layer].values()),
            "tables": {
                name: {
                    "status": v["status"],
                    "file_count": v["file_count"],
                    "row_count": v["row_count"],
                    "total_bytes": v["total_bytes"],
                    "schema_ok": v["schema_ok"],
                }
                for name, v in audit_results[layer].items()
            },
        }
        for layer in ("bronze", "silver", "gold")
    },
    "summary": {
        "total_layers": 3,
        "total_tables_registered": sum(len(v) for v in TABLE_REGISTRY.values()),
        "total_tables_present": sum(
            1 for layer_tbls in audit_results.values() for v in layer_tbls.values() if v["status"] == "PRESENT_VALID"
        ),
        "total_rows_verified": sum(
            max(0, v["row_count"]) for layer_tbls in audit_results.values() for v in layer_tbls.values()
        ),
        "publish_readiness": PUBLISH_READINESS,
        "readiness_blockers": READINESS_BLOCKERS,
        "readiness_warnings": READINESS_WARNINGS,
    },
}

raw_json = json.dumps(manifest_data, indent=2, sort_keys=True, ensure_ascii=False)
manifest_sha256 = hashlib.sha256(raw_json.encode("utf-8")).hexdigest()
manifest_data["manifest_sha256"] = manifest_sha256

target_paths = list(dict.fromkeys([
    WORKING_ROOT / "manifest.json",
    EXPORTS_DIR / "manifest.json",
]))

for p in target_paths:
    p.parent.mkdir(parents=True, exist_ok=True)
    _m_tmp = Path(STAGING_PATH) / "release_manifest.json.tmp"
    _m_tmp.parent.mkdir(parents=True, exist_ok=True)
    _m_tmp.write_text(
        json.dumps(manifest_data, indent=2, sort_keys=True, ensure_ascii=False),
        encoding="utf-8",
    )
    if p.exists():
        p.unlink()
    shutil.copyfile(_m_tmp, p)
    _m_tmp.unlink(missing_ok=True)
    print(f"Manifesto de release salvo em: {p} ({p.stat().st_size:,} bytes)")

print(f"SHA-256 do Manifesto: {manifest_sha256}")

Manifesto de release salvo em: /content/drive/MyDrive/tse-eleicoes-dados/data/manifest.json (7,659 bytes)
SHA-256 do Manifesto: 9d212ef7a86f2e715d2cb50009cd9d5bc9578d2f9a3f5b469f4e32733d2874c9


## Publicação da Nova Versão do Dataset via KaggleHub SDK

Com a validação das **28 tabelas** concluída e o manifesto oficial serializado em `{DATASET_PATH}/manifest.json`, a célula final valida a higiene de `DATASET_PATH` e publica a nova versão consolidada de **`daniellmattos/tse-eleicoes-dados-v2`** utilizando `kagglehub.dataset_upload()` (ou pula o upload quando `TSE_SKIP_KAGGLE_RELEASE=1`).

> `publish_readiness` no manifesto só assume `READY_FOR_RELEASE` quando todas as 28 tabelas (18 Bronze + 5 Silver + 5 Gold) estão `PRESENT_VALID`, com linhas > 0 e zero bloqueios de unicidade ou particionamento.

In [6]:
# ---------------------------------------------------------------------------
# Relatório Final de Prontidão de Release (Checklist em pt-BR)
# ---------------------------------------------------------------------------
bronze_ok = all(v["status"] == "PRESENT_VALID" for v in audit_results["bronze"].values())
silver_ok = all(v["status"] == "PRESENT_VALID" for v in audit_results["silver"].values())
gold_ok = all(v["status"] == "PRESENT_VALID" for v in audit_results["gold"].values())
failed_keys = [
    k for k, v in uniqueness_results.items() if v.get("status") not in ("PASS", "INFO_UPSTREAM_TSE")
]

checklist = [
    ("Integridade da Ingestão das 17 Tabelas Eleitorais Bronze", "Bronze", "APROVADO" if bronze_ok else "FALHA"),
    ("Privacidade e Redação (<ID_REDACTED>) em fact_urna_logs", "Bronze", "APROVADO" if bronze_ok else "FALHA"),
    ("Classificação Semântica v1.1.0 e Carryover em fact_urna_event", "Silver", "APROVADO" if silver_ok else "FALHA"),
    ("Máquina de Estados v1.0.0 em fact_voting_session", "Silver", "APROVADO" if silver_ok else "FALHA"),
    ("Dimensões Conformadas (Eleição / Cargo / Partido)", "Silver", "APROVADO" if silver_ok else "FALHA"),
    ("Camada Gold Enriquecida v2.0.0 (5 Produtos Analíticos)", "Gold", "APROVADO" if gold_ok else "FALHA"),
    ("Unicidade de Chaves Primárias e Naturais Auditada", "Multi-Camada", "APROVADO" if not failed_keys else "FALHA"),
    ("Conformidade de Particionamento Hive (Zero Chaves Proibidas)", "Multi-Camada", "APROVADO" if not partition_audit_issues else "FALHA"),
    ("Reconciliação Quantitativa Bronze -> Silver -> Gold", "Multi-Camada", "APROVADO" if reconcile_status != "DIVERGENTE" else "FALHA"),
    ("Manifesto Oficial Serializado com Hash SHA-256", "Release", "APROVADO"),
]

total_present = manifest_data["summary"]["total_tables_present"]
total_reg = manifest_data["summary"]["total_tables_registered"]
total_rows = manifest_data["summary"]["total_rows_verified"]

print("=" * 94)
print(f"{'TSE ELEIÇÕES 2026 — RELATÓRIO DE PRONTIDÃO DE RELEASE':^94}")
print("=" * 94)
print(f"Versão do Release    : {RELEASE_VERSION}")
print(f"Timestamp UTC        : {TIMESTAMP_UTC}")
print(f"Dataset de Destino   : {DATASET_HANDLE}")
print(f"SHA-256 do Manifesto : {manifest_sha256}")
print(f"Tabelas Auditadas    : {total_present}/{total_reg} tabelas PRESENT_VALID")
print(f"Total Linhas Físicas : {total_rows:,} registros verificados")
print("-" * 94)
print(f"{'ITEM DE VALIDAÇÃO':<62} | {'CAMADA':<13} | {'RESULTADO':<10}")
print("-" * 94)

for item, layer_label, status in checklist:
    print(f"{item:<62} | {layer_label:<13} | [{status}]")

print("=" * 94)
verdict_label = (
    "PRONTO PARA PUBLICAÇÃO DO RELEASE"
    if not READINESS_BLOCKERS and not failed_keys
    else f"BLOQUEADO ({len(READINESS_BLOCKERS)} pendências, {len(failed_keys)} falhas de chave)"
)
print(f"VEREDITO FINAL: [{verdict_label}]")
print("=" * 94)

PHASE_ID = "09_release_pipeline"
PHASE_SUMMARY = "Release multi-camada v2 (28 tabelas Bronze/Silver/Gold): auditoria integral e manifest.json SHA-256"
if READINESS_BLOCKERS or failed_keys:
    raise RuntimeError(
        f"Upload abortado! Release possui bloqueios de prontidão ({READINESS_BLOCKERS}) "
        f"ou falhas de unicidade ({failed_keys})."
    )
forbidden_extensions = {".zip", ".jez", ".tmp", ".part", ".log"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and any(sfx.lower() in forbidden_extensions for sfx in p.suffixes)
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if is_directory_empty(Path(DATASET_PATH)):
    raise RuntimeError(
        f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
    )

if (
    globals().get("__name__") == "__main__"
    and not os.getenv("PYTEST_CURRENT_TEST")
    and not os.getenv("TSE_SKIP_KAGGLE_RELEASE")
):
    import kagglehub

    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")

                    TSE ELEIÇÕES 2026 — RELATÓRIO DE PRONTIDÃO DE RELEASE                     
Versão do Release    : 2.0.0
Timestamp UTC        : 2026-11-28T02:47:06.342124+00:00
Dataset de Destino   : daniellmattos/tse-eleicoes-dados-v2
SHA-256 do Manifesto : 9d212ef7a86f2e715d2cb50009cd9d5bc9578d2f9a3f5b469f4e32733d2874c9
Tabelas Auditadas    : 28/28 tabelas PRESENT_VALID
Total Linhas Físicas : 13,212,814,890 registros verificados
----------------------------------------------------------------------------------------------
ITEM DE VALIDAÇÃO                                              | CAMADA        | RESULTADO 
----------------------------------------------------------------------------------------------
Integridade da Ingestão das 17 Tabelas Eleitorais Bronze       | Bronze        | [APROVADO]
Privacidade e Redação (<ID_REDACTED>) em fact_urna_logs        | Bronze        | [APROVADO]
Classificação Semântica v1.1.0 e Carryover em fact_urna_event  | Silver        | [APROVADO]
Máqu